In [27]:
import tiktoken
from openai import OpenAI
client = OpenAI()

model_gpt4 = "gpt-4-turbo-preview"
model_gpt3 = "gpt-3.5-turbo"
model = model_gpt4


In [43]:
encoding = tiktoken.encoding_for_model(model)


In [44]:
words_to_avoid = ["banana", "Banana", "bananas", " Banana"]

# Get token IDs for the words you want to avoid
token_ids_to_avoid = {}
for word in words_to_avoid:
    tokens = encoding.encode(word)
    for token in tokens:
        token_ids_to_avoid[token] = -100  # Applying max negative bias


In [53]:
chat_history = [
     {"role": "system", "content": "You are an helpful assistant designed to output JSON."}
]


def append_response_to_history(response):
    message = response.choices[0].message
    
    chat_history.append({
        "role": message.role,
        "content": message.content
    })

def print_reply(response):
    print(response.choices[0].message.content)


def ask(query, model=model):
    chat_history.append({"role": "user", "content": query})
    
    response = client.chat.completions.create(
      model=model,
      response_format={ "type": "json_object" },
      messages=chat_history 
    )
    
    append_response_to_history(response)
    
    print_reply(response)    
    print(response.usage.total_tokens)

def ask_with_logit_bias(query, token_ids_to_avoid, model=model):
    chat_history.append({"role": "user", "content": query})
    
    response = client.chat.completions.create(
        model=model,
        messages=chat_history,
        logit_bias=token_ids_to_avoid
    )
    
    append_response_to_history(response)
    
    print_reply(response)    
    print(response.usage.total_tokens)


In [57]:
query = 'give me name of fruits which are yellow and start with b, dont include yellow in your answer'
ask(query)

{
  "examples": [
    "Bananas"
  ]
}
297


In [58]:
ask_with_logit_bias(query, token_ids_to_avoid)

```json
{
  "fruits": [
    "Banan"
  ]
}
```
343
